In [ ]:
import pandas as pd
from kfre import kfre_person

In [ ]:
image_path_png = "../images/kfre_performance.png"
image_path_svg = "../images/kfre_performance.svg"

## Single Patient Risk Calculation

In [ ]:
risk_percentage = kfre_person(
    age=57.28,
    is_male=False,
    eGFR=15.0,
    uACR=1762.001840,
    is_north_american=False,
    years=2,
    dm=None,
    htn=None,
    albumin=None,
    phosphorous=None,
    bicarbonate=None,
    calcium=None
) * 100  # Convert to percentage

message = f"The 2-year risk of kidney failure for this patient is"
print(f"{message} {risk_percentage:.2f}%.")

### Example Calculation for 2-year and 5-year Risk

In [ ]:
for years in [2, 5]:
    risk_percentage = (
        kfre_person(
            age=57.28,
            is_male=False,  # is the patient male?
            eGFR=15.0,  # ml/min/1.73 m^2
            uACR=1762.001840,  # mg/g
            is_north_american=False,  # is the patient from North America?
            years=years,
            ################################################################
            # Uncomment "dm" and "htn" for the 6-variable model:
            ################################################################
            # dm=0,
            # htn=1,
            ################################################################
            # Comment out "dm" and "htn"; uncomment the following lines for
            # the 8-variable model:
            ################################################################
            # albumin=3.0, # g/dL
            # phosphorous=3.162, # mg/dL
            # bicarbonate=21.3, # mEq/L
            # calcium=9.72, # mg/dL
        )
        * 100  # multiply by 100 to convert to percentage
    )

    message = f"The {years}-year risk of kidney failure for this patient is"
    print(f"{message} {risk_percentage:.2f}%.")

## Conversion of Clinical Parameters

In [ ]:
from kfre import perform_conversions

In [ ]:
df = pd.read_csv("../data/12882_2021_2402_MOESM8_ESM.csv")

In [ ]:
# Perform conversions using the wrapper function, specifying all parameters
# and specify new column names
df = perform_conversions(
    df=df,
    reverse=False,
    convert_all=True,
)

# Print the DataFrame to see the changes
df

In [ ]:
from kfre import upcr_uacr

In [ ]:
df["uACR"] = upcr_uacr(
    df=df,
    sex_col="SEX",
    diabetes_col="Diabetes (1=yes; 0=no)",
    hypertension_col="Hypertension (1=yes; 0=no)",
    upcr_col="uPCR_mg_g",
    female_str="Female",
)

In [ ]:
print(df["uACR"])

## Classifying ESRD Outcomes

In [ ]:
from kfre import class_esrd_outcome

In [ ]:
# 2-year outcome
df = class_esrd_outcome(
    df=df,
    col="ESRD",
    years=2,
    duration_col="Follow-up YEARS",
    prefix="esrd",
    create_years_col=False,
)

# 5-year outcome
df = class_esrd_outcome(
    df=df,
    col="ESRD",
    years=5,
    duration_col="Follow-up YEARS",
    prefix="esrd",
    create_years_col=False,
)

In [ ]:
df

### Batch Risk Calculation for Multiple Patients

In [ ]:
from kfre import add_kfre_risk_col

In [ ]:
# THEN compute predictions
df = add_kfre_risk_col(
    df=df,
    age_col="Age",
    sex_col="SEX",
    eGFR_col="eGFR-EPI",
    uACR_col="uACR",
    dm_col="Diabetes (1=yes; 0=no)",
    htn_col="Hypertension (1=yes; 0=no)",
    albumin_col="Albumin_g_dl",
    phosphorous_col="Phosphate_mg_dl",
    bicarbonate_col="Bicarbonate (mmol/L)",
    calcium_col="Calcium_mg_dl",
    num_vars=[4, 6, 8],
    years=(2, 5),
    is_north_american=False,
    copy=False,
)

## Performance Assessment

In [ ]:
from kfre import plot_kfre_metrics, eval_kfre_metrics

In [ ]:
# Check what outcome columns exist
print([c for c in df.columns if 'year_outcome' in c])

# Check what kfre prediction columns exist  
print([c for c in df.columns if 'kfre_' in c])

# Check NaNs in those specific columns
outcome_cols = [c for c in df.columns if 'year_outcome' in c]
kfre_cols = [c for c in df.columns if 'kfre_' in c]
print(df[outcome_cols + kfre_cols].isnull().sum())

In [ ]:
kfre_cols = [c for c in df.columns if 'kfre_' in c]
df = df.dropna(subset=kfre_cols)

metrics_df_n_var = eval_kfre_metrics(
    df=df,
    n_var_list=[4, 6, 8],
    outcome_years=[2, 5],
)
print(metrics_df_n_var)

In [ ]:
from kfre import plot_kfre_metrics

In [ ]:
plot_kfre_metrics(
    df=df,                       # DataFrame to produce plots for
    num_vars=[4, 6, 8],          # 4,6,8 KFRE variables
    figsize=[6, 6],             # Custom figure size
    mode="plot",                 # Can be 'prep', 'plot', or 'both'
    image_prefix="performance",  # Optional prefix for saved images
    bbox_inches="tight",         # Bounding box in inches for the saved images
    plot_type="all_plots",       # Can be 'auc_roc', 'precision_recall', or 'all_plots'
    show_years=[2, 5],           # Year outcomes to show in the plots
    plot_combinations=True,      # Plot combinations of all variables in one plot
    show_subplots=True,          # Place all plots on one subplot; False does individual
    decimal_places=3,            # Number of decimal places in legend
    image_path_png = image_path_png,
    image_path_svg = image_path_svg,
    image_filename = "kfre_performance"
)

In [ ]:
import matplotlib.pyplot as plt
from kfre import class_ckd_stages

# Assign CKD stages
df = class_ckd_stages(
    df=df,
    egfr_col="eGFR-EPI",
    stage_col="stage",
    combined_stage_col="stage_combined"
)

import os
os.makedirs("figures", exist_ok=True)

# Figure 1: Risk by CKD stage
fig, ax = plt.subplots(figsize=(8, 6))
stage_order = ["CKD Stage 4", "CKD Stage 5"]
data_by_stage = [
    df[df["stage"] == s]["kfre_4var_2year"].dropna().clip(lower=0)
    for s in stage_order
]
ax.boxplot(data_by_stage, tick_labels=["Stage 4", "Stage 5"])
ax.set_ylim(bottom=0)
ax.set_xlabel("CKD Stage")
ax.set_ylabel("Predicted 2-Year Risk")
ax.set_title("KFRE 4-Variable, 2-Year Risk by CKD Stage")
plt.tight_layout()
plt.savefig("figures/risk_by_stage.svg")
plt.show()

# Figure 2: 4-var vs 8-var scatter
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(df["kfre_4var_2year"], df["kfre_8var_2year"], alpha=0.5, s=10)
lims = [0, max(df["kfre_4var_2year"].max(), df["kfre_8var_2year"].max())]
ax.plot(lims, lims, "--", color="gray")
ax.set_xlabel("4-Variable KFRE (2-year)")
ax.set_ylabel("8-Variable KFRE (2-year)")
ax.set_title("4- vs. 8-Variable KFRE: 2-Year Risk Estimates")
plt.tight_layout()
plt.savefig("figures/4var_vs_8var.svg")
plt.show()

In [ ]:
print(df["stage"].value_counts())